# Apache Cassandra

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Apache Cassandra is a distributed, masterless, wide-column store built for high write throughput and linear horizontal scale. Data is partitioned across a peer-to-peer ring, replicated for durability, and queried with CQL using a table design that starts from the query rather than from normalization.

## Contents

<details>
<summary>35 sections - click to expand</summary>

- [Explain what is Cassandra?](#explain-what-is-cassandra)
- [How do you model data in Cassandra?](#how-do-you-model-data-in-cassandra)
- [What is Cassandra Data Model?](#what-is-cassandra-data-model)
- [Explain what is a Keyspace in Cassandra?](#explain-what-is-a-keyspace-in-cassandra)
- [What are the different composite keys in Cassandra?](#what-are-the-different-composite-keys-in-cassandra)
- [What is mandatory while creating a table in Cassandra?](#what-is-mandatory-while-creating-a-table-in-cassandra)
- [How to write a Query in Cassandra?](#how-to-write-a-query-in-cassandra)
- [What is Cql?](#what-is-cql)
- [How does Cassandra write?](#how-does-cassandra-write)
- [What do you mean by commit log in Cassandra?](#what-do-you-mean-by-commit-log-in-cassandra)
- [Define Memtable?](#define-memtable)
- [What is SStable and how is it different from other relational tables?](#what-is-sstable-and-how-is-it-different-from-other-relational-tables)
- [Explain the concept of compaction in Cassandra?](#explain-the-concept-of-compaction-in-cassandra)
- [Explain the concept of Bloom Filter?](#explain-the-concept-of-bloom-filter)
- [Explain what is a Column Family in Cassandra?](#explain-what-is-a-column-family-in-cassandra)
- [What is the use of Cassandra CQL collection?](#what-is-the-use-of-cassandra-cql-collection)
- [Explain what is a Cluster in Cassandra?](#explain-what-is-a-cluster-in-cassandra)
- [State the differences between Node and Cluster And DataCenter in Cassandra?](#state-the-differences-between-node-and-cluster-and-datacenter-in-cassandra)
- [What is data replication in Cassandra?](#what-is-data-replication-in-cassandra)
- [Can we change Replication Factor on a Live Cluster?](#can-we-change-replication-factor-on-a-live-cluster)
- [Explain Cap Theorem?](#explain-cap-theorem)
- [What are consistency levels in Cassandra?](#what-are-consistency-levels-in-cassandra)
- [Explain the concept of tunable consistency in Cassandra?](#explain-the-concept-of-tunable-consistency-in-cassandra)
- [Does Cassandra support ACID transactions?](#does-cassandra-support-acid-transactions)
- [Explain how Cassandra delete data?](#explain-how-cassandra-delete-data)
- [Explain Tombstone in Cassandra?](#explain-tombstone-in-cassandra)
- [What is difference between Column and Super Column?](#what-is-difference-between-column-and-super-column)
- [Explain what is Cassandra cqlsh?](#explain-what-is-cassandra-cqlsh)
- [Mention what needs to be taken care while adding a Column?](#mention-what-needs-to-be-taken-care-while-adding-a-column)
- [What are the trade-offs and when should you not use Cassandra?](#what-are-the-trade-offs-and-when-should-you-not-use-cassandra)
- [Things to remember: key numbers and limits](#things-to-remember-key-numbers-and-limits)
- [Things to remember: semantics to memorize](#things-to-remember-semantics-to-memorize)
- [Things to remember: mistakes that keep coming back](#things-to-remember-mistakes-that-keep-coming-back)
- [Real world use cases for Cassandra](#real-world-use-cases-for-cassandra)
- [Cassandra exercises](#cassandra-exercises)

</details>

Tip: the links open the sections on GitHub; in Jupyter or Colab, use the notebook's outline panel in the sidebar instead.

## Explain what is Cassandra?

Cassandra is an open source data storage system developed at Facebook for inbox search and designed for storing and managing large amounts of data across commodity servers. It can serve as both a real-time data store for online applications and as a read-intensive database for business intelligence systems.

Architecturally it combines two papers: Amazon Dynamo's peer-to-peer ring, consistent hashing, and tunable consistency, and Google Bigtable's storage model (commit log, memtable, SSTable) with a column-family data model. There is no master: every node is a peer, any node can serve any request, and data is placed by the partitioner and the replication strategy.

```text
client -> coordinator node -> replicas (RF copies)
              |
              +-- commit log -> memtable -> flush to SSTable -> compaction
```

Key properties: linear scalability by adding nodes, no single point of failure, tunable consistency per query, multi-datacenter replication, and a schema-optional column model with a SQL-like language (CQL). It is an AP system: availability and partition tolerance are prioritized, and consistency is chosen per operation.

*Also asked as: Explain Cassandra.*

The ring is a consistent-hash circle. The partitioner (Murmur3 by default) hashes the partition key to a token in the range -2^63 to 2^63-1, and each node owns one or more token ranges. To place a partition, Cassandra takes its token and walks clockwise, choosing the first node as the primary replica and continuing to the next distinct racks or data centers until RF copies are placed. No coordinator is special: any client or node computes the same placement from the same ring metadata.

Virtual nodes (vnodes) make ownership fine-grained. Instead of one large range per node, a node claims many small ranges (`num_tokens`, 16 by default in Cassandra 4.0 and later). Many small ranges mean a new node takes a slice from many peers at once, so the ring streams in parallel and a failed node's load spreads across the cluster instead of onto one neighbor.

Worked example: with 6 nodes and RF=3, a partition whose token falls just after node B's range end is stored on C, D and E, the next three distinct nodes clockwise. Common mistake: after adding a node, forgetting `nodetool cleanup` on the old nodes, which leaves data they no longer own on disk and makes the cluster look unbalanced while reads still work.

## How do you model data in Cassandra?

Cassandra modeling starts with queries, not normalization. You design tables around access patterns because joins are not the normal path. A table should have a partition key that spreads data evenly and clustering columns that support the required sort/order within a partition.

Example:

```sql
create table orders_by_customer (
  customer_id text,
  order_ts timestamp,
  order_id text,
  amount decimal,
  primary key ((customer_id), order_ts, order_id)
) with clustering order by (order_ts desc);
```

This table is good for "show recent orders for one customer." It is not good for arbitrary analytics across all customers; that belongs in Spark, a warehouse, or another table designed for that query.

The method: write down each query, annotate its equality filters and its sort, make the equality column the partition key and the sort column the leading clustering column, then build one table per query. Duplicate data deliberately and accept that updates to duplicated copies are the application's job.

```sql
-- Query: orders for one customer, newest first (partition bounded by customer)
create table orders_by_customer (
  customer_id text,
  order_ts timestamp,
  order_id text,
  amount decimal,
  primary key ((customer_id), order_ts, order_id)
) with clustering order by (order_ts desc);

-- Query: today's orders for one tenant, newest first (partition bounded by tenant+day)
create table orders_by_tenant_day (
  tenant_id text,
  day date,
  order_ts timestamp,
  order_id text,
  amount decimal,
  primary key ((tenant_id, day), order_ts, order_id)
) with clustering order by (order_ts desc);
```

Two failure modes to name explicitly: an unbounded partition (for example `primary key ((customer_id), order_ts)` when a customer can have millions of orders) and a table that only works with `ALLOW FILTERING`, which tells you the table does not match the query. Secondary indexes in Cassandra are local and low-cardinality only; they are not a substitute for a purpose-built table.

The partition key does three jobs at once: it groups the rows that are read together, it decides the token, and the token decides which replicas store the data. Cardinality matters because a partition lives entirely on one replica set. A low-cardinality key (a boolean, a status, a country) puts most of the table on a handful of nodes. Aim for many distinct values that are also non-monotonic.

Worked example: an events table queried by tenant and time range. `primary key (ts)` is wrong twice over: every write goes to the newest token, creating a hot partition, and no read is scoped to a tenant. `primary key ((tenant_id, day), ts)` gives each read one bounded partition and spreads writes across tenants and days. Common mistake: `primary key (event_id)`, which distributes writes perfectly but turns every list query into a full scan.

## What is Cassandra Data Model?

The Cassandra Data Model consists of four main components:

+ Cluster: Made up of multiple nodes and keyspaces
+ Keyspace: a namespace to group multiple tables, with a replication strategy and replication factor
+ Column: consists of a column name, value and timestamp
+ Table (formerly ColumnFamily): multiple columns with row key reference.

The logical hierarchy is cluster > keyspace > table > partition > row > column, and the physical hierarchy is node > data directory > SSTable files. The row key is a partition key plus any clustering columns, so "column family" is really a sorted map of partitions, each holding a sorted set of rows.

```sql
-- Logical hierarchy expressed in CQL
create keyspace if not exists shop
  with replication = {'class': 'NetworkTopologyStrategy', 'dc1': 3, 'dc2': 3}
  and durable_writes = true;

use shop;

create table cart_items (
  user_id uuid,
  cart_id timeuuid,
  added_at timestamp,
  sku text,
  qty int,
  primary key ((user_id), cart_id)
);
```

The older Thrift vocabulary (column families, super columns, comparators, validators) is historical; modern Cassandra uses CQL tables and the storage engine still implements them as sorted maps of cells with per-cell timestamps.

*Also asked as: Mention what are the main components of Cassandra data model? List out the other components of Cassandra?*

The partition is the unit that matters. It is stored on one replica set, written atomically, and read in one seek pattern; the rows inside it are sorted by the clustering columns. Two rows that share a partition key and differ in the clustering value are two rows in one partition. That is what "wide row" means, and an unbounded partition is a design bug rather than a feature.

The primary key is unique, so an `INSERT` with an existing key is an upsert, not an error. Every row in a partition repeats the partition key logically but stores it once, and a `static` column holds one value for the whole partition. The data model is therefore a sorted map of partitions, each a sorted set of rows, each row a sparse set of typed cells.

## Explain what is a Keyspace in Cassandra?

In Cassandra, a keyspace is a namespace that determines data replication on nodes. A keyspace is not per node; a cluster typically has several keyspaces, each with its own replication strategy and replication factor, and every table belongs to exactly one keyspace.

```sql
create keyspace iot
  with replication = {'class': 'NetworkTopologyStrategy', 'dc1': 3}
  and durable_writes = true;

alter keyspace iot
  with replication = {'class': 'NetworkTopologyStrategy', 'dc1': 3, 'dc2': 3};

describe keyspaces;
describe keyspace iot;
```

`SimpleStrategy` (a single RF, ignore data centers) is for development only; production multi-DC deployments use `NetworkTopologyStrategy`. `durable_writes = false` disables the commit log for that keyspace and is almost never what you want.

*Also asked as: What is keyspace in Cassandra?*

Each keyspace carries its own replication strategy and factor, so one cluster can hold a 3-replica operational keyspace and a 1-replica scratch keyspace. The partitioner is a node setting, not a keyspace setting, so every keyspace in a cluster uses the same token space. `DESCRIBE KEYSPACE iot` and `SELECT * FROM system_schema.keyspaces` show the replication metadata.

## What are the different composite keys in Cassandra?

In Cassandra, composite keys are used to define a key or a column name with a concatenation of data of different type. There are two types of composite key in Cassandra:

+ Row Key (partition key)
+ Column Name (clustering columns)

A composite partition key is written `primary key ((a, b), c)`: `a` and `b` together decide the partition, and `c` decides the sort order and uniqueness inside it.

```sql
-- Composite partition key: (tenant_id, day) locates the partition
-- Clustering columns: order_ts then order_id order rows inside it
create table orders (
  tenant_id text,
  day date,
  order_ts timestamp,
  order_id text,
  amount decimal,
  primary key ((tenant_id, day), order_ts, order_id)
);

-- Composite clustering: events of one user sorted by time, then id
create table user_events (
  user_id text,
  event_ts timestamp,
  event_id timeuuid,
  payload text,
  primary key ((user_id), event_ts, event_id)
);
```

The partition key determines which node stores the data (via the partitioner and the token ring); the clustering columns determine on-disk sort order within a partition and are what make range queries cheap. The tuple serialization of the composite key is what allows several columns to behave as one key.

*Also asked as: Explain what is composite type in Cassandra?*

Clustering columns are consulted left to right. A query must pin every partition key column with equality, and may then restrict clustering columns in order: it can filter `ck1`, or `ck1` and `ck2`, but not `ck2` alone. That rule is what makes a composite clustering key a usable sorted index instead of a general-purpose filter.

Worked example on `primary key ((tenant_id, day), order_ts, order_id)`: `WHERE tenant_id=? AND day=?` is a single-partition read. Adding `AND order_ts >= ?` turns it into a range scan inside that partition. Adding `AND order_id=?` without `order_ts` is rejected. Common mistake: thinking of a composite partition key as independent keys - you must supply every column of `((a, b), c)`'s `a` and `b` or the query cannot be routed.

## What is mandatory while creating a table in Cassandra?

While creating a table primary key is mandatory, it is made up of one or more columns of a table. The primary key is the partition key, optionally followed by clustering columns, and it is the only way to address rows directly. Cassandra has no supporting indexes by default beyond the primary key, so the key is the schema.

```sql
create table readings (
  sensor_id text,
  ts timestamp,
  value double,
  primary key (sensor_id, ts)
);
-- primary key (sensor_id, ts) == partition key sensor_id, clustering ts

create table readings_by_day (
  sensor_id text,
  day date,
  ts timestamp,
  value double,
  primary key ((sensor_id, day), ts)
) with default_time_to_live = 7776000;   -- 90 days, in seconds
```

A primary key is required; everything else - clustering order, TTL, compaction strategy, caching, compression - is a table option. Choosing a bad primary key cannot be fixed later without creating a new table and migrating, which is why key design is the first interview question for any Cassandra modeling task.

The primary key is the only required clause. CQL has no `NOT NULL`, no `DEFAULT`, and no unique constraint other than the primary key. A non-key column may be `static`, meaning one value shared by every row in the partition - useful for a per-partition marker or counter, but still one cell served by one replica set.

## How to write a Query in Cassandra?

Using CQL (Cassandra Query Language). cqlsh is used for interacting with the database.

```bash
# Start cqlsh against a node
cqlsh 10.0.0.11 9042 -u app -p secret -k iot

# Execute a file of statements, and capture output to a file
cqlsh -f schema.cql
cqlsh -e "select release_version from system.local"
```

```sql
-- Insert, upsert, select, update, delete: all CQL DML
insert into readings (sensor_id, ts, value) values ('s-1', '2026-10-04T10:00:00Z', 21.7);

select sensor_id, ts, value
from readings
where sensor_id = 's-1'
  and ts >= '2026-10-04T00:00:00Z'
  and ts <  '2026-10-05T00:00:00Z'
order by ts desc
limit 100;

update readings set value = 22.1 where sensor_id = 's-1' and ts = '2026-10-04T10:00:00Z';

delete from readings where sensor_id = 's-1' and ts = '2026-10-04T10:00:00Z';
```

Rules that keep queries efficient: the partition key must be restricted with equality, clustering columns may then be restricted with equality or ranges in order, and any other predicate raises "Cannot execute this query ... use ALLOW FILTERING". `ALLOW FILTERING` scans every partition on every node and should be treated as a bug in a design review, not a fix.

Reads obey the same key rules as writes. Add `LIMIT`, because an unbounded `SELECT` pages the driver through the whole partition, and prefer prepared statements so the driver caches the query plan and the token routing. A read without a partition key, or with a predicate on a non-key column, becomes a cluster-wide scan behind `ALLOW FILTERING`.

## What is Cql?

CQL is Cassandra Query Language to access and query the Apache distributed database. It consists of a CQL parser that hides all the implementation details to the server. The syntax of CQL is similar to SQL but it does not alter the Cassandra data model: there are no joins, no subqueries in the general case, and no foreign keys, and CQL is not SQL.

```sql
-- CQL looks like SQL but the semantics are different
select * from orders_by_customer where customer_id = 'C1' limit 10;   -- OK
select * from orders_by_customer where amount > 100;                  -- needs ALLOW FILTERING

-- Collections and UDTs have no SQL equivalent
create type address (street text, city text, zip text);
create table users (
  user_id uuid primary key,
  emails set<text>,
  home frozen<address>,
  prefs map<text, text>
);
```

Things CQL deliberately does not have: joins, aggregates across partitions that scale (standard `count(*)` on a big table scans all nodes), transactions in the ACID sense, and stored procedures. The one escape hatch is lightweight transactions (`IF NOT EXISTS`, `IF EXISTS` conditions), which add Paxos rounds and latency.

*Also asked as: Which query language is used in Cassandra database?*

CQL is schema-first and has upsert `INSERT`, no joins, and no stored procedures. `SELECT` supports `count(*)`, `min`, `max`, `sum`, `avg`, `group by` inside a partition, and `JSON` output, but an aggregate without a partition key restriction scans every node. `IF NOT EXISTS` and `IF <condition>` turn a write into a Paxos-backed compare-and-set with a real latency cost.

## How does Cassandra write?

Cassandra performs the write function by applying two commits: first it writes to a commit log on disk and then commits to an in-memory structure known as memtable. Once the two commits are successful, the write is achieved. Writes are later written in the table structure as SSTable (sorted string table). Cassandra offers speedier write performance.

The full path for a write sent to any node (the coordinator):

```text
1. coordinator computes the token(s) for the partition key
2. coordinator forwards to the RF replicas for that token
3. each replica appends the mutation to its commit log (durability)
4. each replica writes it to its memtable (in-memory, sorted by key)
5. coordinator waits for the requested consistency level, then acknowledges
6. later: memtable is flushed to an immutable SSTable on disk
7. later: compaction merges SSTables and drops overwritten/tombstoned data
```

```bash
# Watch the write path in action
nodetool flush iot readings          # force memtable -> SSTable
nodetool drain                       # flush and stop accepting writes
nodetool tablestats iot.readings     # SSTable count, memtable size, write latency
```

Writes are append-only: there is no read-modify-write, no in-place update, and therefore no lock contention. That is why a Cassandra write is a sequential commit log append plus a memory write, and why write throughput scales with node count rather than with disk seek time.

*Also asked as: Explain how Cassandra writes data? How does Cassandra perform write function?*

Write path detail: the coordinator hashes the key, finds the RF replicas, and sends the mutation to all of them in parallel. Each replica appends to its commit log, writes into its memtable, and acknowledges. The coordinator returns success once the requested consistency level has acknowledged, not when every replica has. A replica that is down receives a hint, which is replayed when it returns, for outages shorter than `max_hint_window_in_ms` (3 hours by default).

Worked example with RF=3 and `CL=QUORUM`: the coordinator waits for 2 of 3 replica acks. One replica down still succeeds and stores a hint; two replicas down fails the write. Common mistake: reading back immediately at `CL=ONE` from a replica that did not acknowledge, then concluding the write was lost. With `R + W <= RF` a read may legitimately miss recent writes.

## What do you mean by commit log in Cassandra?

In Cassandra, commit log is a crash-recovery mechanism. Every write operation is written to the commit log before it is acknowledged, and the changed data is appended, not updated in place, so the log is a sequential write.

```bash
nodetool flush <keyspace> <table>   # forces memtables to disk; commitlog segments become reclaimable
nodetool drain                      # flush all memtables, stop the commitlog, prepare for shutdown
```

Recovery works by replaying commit log segments that had no matching flush. Segments are recycled once every mutation in them has been flushed to an SSTable. Key settings are `commitlog_sync` (`periodic` by default, `batch` for fsync-per-write), `commitlog_sync_period`, and `commitlog_total_space_in_mb`, which controls how aggressively dirty memtables are flushed when the log fills.

*Also asked as: Explain how Cassandra writes changed data into Commitlog?*

The commit log is a set of preallocated segments written sequentially and recycled only when every mutation in a segment has been flushed to an SSTable. `commitlog_sync: periodic` (the default) fsyncs on the interval in `commitlog_sync_period_in_ms`; `commitlog_sync: batch` fsyncs before acknowledging, which costs latency but does not lose an acknowledged write on power loss. On restart Cassandra replays segments that have no matching flush.

## Define Memtable?

Similar to a table, memtable is an in-memory/write-back cache space consisting of content in key and column format. The data in memtable is sorted by key, and each table (column family) has a distinct memtable that retrieves column data via key. It stores the writes until it is full, and then flushed out.

```bash
nodetool flush iot readings              # memtable -> SSTable now
nodetool tablestats iot.readings | grep -i memtable
```

A memtable is flushed when it exceeds `memtable_cleanup_threshold` (derived from total memory), when the commit log is full, or when `nodetool flush` is issued. Reads must merge the memtable with all SSTables on disk, which is why many live SSTables degrade read latency and why compaction matters.

*Also asked as: Explain what is Memtable in Cassandra?*

Each table has its own memtable, held off-heap by default in modern releases. A memtable is flushed when it passes `memtable_cleanup_threshold` (a share of the configured memtable budget), when the commit log fills, or on `nodetool flush`. Until it is flushed, every read of that table must merge the memtable with every SSTable. Many small flushes create many small SSTables, which is exactly the read amplification that compaction exists to fix.

## What is SStable and how is it different from other relational tables?

SSTable expands to 'Sorted String Table', which refers to an important data file in Cassandra and accepts regular written memtables. They are stored on disk and exist for each Cassandra table. Exhibiting immutability, SStables do not allow any further addition and removal of data items once written. For each SSTable, Cassandra creates three separate files like partition index, partition summary and a bloom filter.

That immutability is the key difference from a relational table: updates and deletes are new records (writes or tombstones), never in-place edits, and compaction later reconciles them. Unlike a relational table, an SSTable is sorted by token, then by clustering columns, and has no notion of rows across partitions being contiguous.

```bash
# Inspect the files that make up an SSTable set
ls -1 /var/lib/cassandra/data/iot/readings-*/ | head
# *-Data.db  *-Index.db  *-Filter.db  *-Summary.db  *-Statistics.db  *-CompressionInfo.db

# Dump an SSTable to readable JSON (offline tool)
sstabledump /var/lib/cassandra/data/iot/readings-*/mc-1-big-Data.db | head -40

# Per-table SSTable and read/write statistics
nodetool tablestats iot.readings
```

`*-Index.db` holds the partition index, `*-Summary.db` a sampled summary of that index, `*-Filter.db` the Bloom filter, and `*-Statistics.db` the metadata and min/max timestamps used for compaction.

*Also asked as: What is SSTable? How the SSTable is different from other relational tables?*

An SSTable is immutable and sorted by token, then by clustering columns. A set of files makes one SSTable: `Data.db` (the cells), `Index.db` (the partition index, token to file offset), `Summary.db` (a sampled sparse index of that index, so the whole index is not read), `Filter.db` (the Bloom filter), `Statistics.db` (metadata), and `CompressionInfo.db` (chunk offsets). A point read narrows candidates with the filter and summary, then reads only the matching chunks.

## Explain the concept of compaction in Cassandra?

Compaction refers to a maintenance process in Cassandra in which the SSTables are reorganized for data optimization of the data structures on the disk. The compaction process is useful when interacting with the memtable. There are two types of compaction in Cassandra:

+ Minor compaction: started automatically when a new SSTable is created. Here, Cassandra condenses all the equally sized SSTables into one.
+ Major compaction: is triggered manually using nodetool. Compacts all SSTables of a table into one (avoid on large tables: it needs free space equal to the table size and cannot be interrupted).

Modern compaction strategies are chosen per table:

+ SizeTieredCompactionStrategy (STCS): write-optimized default; groups similarly sized SSTables. Good for write-heavy workloads without time locality.
+ LeveledCompactionStrategy (LCS): read-optimized, keeps SSTables in levels with non-overlapping ranges; higher write amplification, predictable read latency.
+ TimeWindowCompactionStrategy (TWCS): groups SSTables by time window; ideal for time series where data expires by TTL and old windows are dropped whole.
+ UnifiedCompactionStrategy (UCS, newer releases): a single strategy that adapts across the size/level/time spectrum.

```sql
-- Time series: one SSTable set per day, TTL expiry drops whole windows
alter table readings_by_day
  with compaction = {
    'class': 'TimeWindowCompactionStrategy',
    'compaction_window_unit': 'DAYS',
    'compaction_window_size': 1
  }
  and gc_grace_seconds = 864000;
```

```bash
nodetool compact iot readings                    # major compaction (use with care)
nodetool compactionstats                         # progress of running compactions
nodetool setcompactionthroughput 128             # MB/s cap
nodetool garbagecollect iot readings             # remove overwritten/tombstoned data aggressively
```

Write amplification is the cost: STCS writes each byte a few times, LCS more, so compaction tuning is the main lever on both write throughput and read latency.

Strategies in practice. STCS merges similarly sized SSTables and is the write-optimized default, but it can leave data fragmented across many SSTables. LCS keeps a leveled tree with non-overlapping ranges, so a read touches roughly one SSTable per level; it pays much more write amplification. TWCS groups SSTables into time windows and compacts or drops a window as a unit, which makes TTL expiry cheap. UCS (Cassandra 5.0) replaces the fixed strategies with one adaptive strategy.

Worked example: a time-series table with TTL 90 days and TWCS windows of 1 day drops an expired window whole, with no tombstone scan. Common mistake: running `nodetool compact` (a major compaction) on a multi-terabyte table. It needs free disk roughly equal to the table size, rewrites everything at once, and cannot be paused cleanly. Watch `nodetool compactionstats` and cap throughput with `nodetool setcompactionthroughput`.

## Explain the concept of Bloom Filter?

Associated with an SSTable, Bloom filter is an off-heap (off the Java heap to native memory) data structure to check whether there is any data available in the SSTable before performing any I/O disk operation. A Bloom filter can say "definitely not here" or "maybe here", never "definitely here", so a false positive costs one wasted disk read but a false negative never happens.

```bash
# Bloom filter effectiveness: recent bloom filter false positives/negatives
nodetool tablestats iot.readings | grep -iA3 bloom
```

Tune with `bloom_filter_fp_chance` per table. The default (0.1 with SizeTiered, 0.01 with Leveled) trades memory for fewer wasted reads: a lower value means a larger filter in off-heap memory. For a table with millions of partitions and point lookups, the Bloom filter is what prevents every read from touching every SSTable.

*Also asked as: What is the use of Bloom Filter in Cassandra?*

Lookup order for a point read: compute the token, ask each SSTable's Bloom filter (off-heap) whether the partition can be there, use the partition summary to jump into the partition index, seek to the offset in the data file, then merge whatever the memtable and surviving SSTables return. A false positive costs one wasted disk seek. A lower `bloom_filter_fp_chance` shrinks that waste at the cost of more off-heap memory.

## Explain what is a Column Family in Cassandra?

Column family in Cassandra refers to a collection of rows, and it is the legacy name for what CQL calls a table. A column family is a partitioned, sorted map: partitions are located by key, and inside each partition columns (cells) are sorted by clustering key with a value and timestamp each.

```sql
create table events_by_user (
  user_id text,
  event_ts timestamp,
  event_type text,
  primary key ((user_id), event_ts)
);
-- "column family" events_by_user: partitions keyed by user_id, rows sorted by event_ts
```

The difference from a relational table is that rows in a column family need not share the same set of columns: different partitions can carry different (and later-added) columns, and the set of columns within a row is sparse. Two tables can hold the same data in different orders - the classic Cassandra denormalization - because there is no join to fall back on.

*Also asked as: What is ColumnFamily? What do you mean by column family in Cassandra?*

Modern vocabulary: a column family is a CQL table, and a row identity is the partition key plus the clustering columns. Two tables can hold the same data in different orders because there is no join. The application writes both tables, and a repair or backfill is what keeps them consistent when a write is retried or lost.

## What is the use of Cassandra CQL collection?

Cassandra CQL collection is used to collect the data and store it in a column where each collection represents the same type of data. CQL consists of three types of collections:

+ SET: an unordered list of unique elements.
+ LIST: elements arranged in an order, and it can contain duplicate values.
+ MAP: a collection of unique elements in the form of key-value pairs.

```sql
create table users (
  user_id uuid primary key,
  emails set<text>,
  phone_numbers list<text>,
  prefs map<text, text>
);

update users set emails = emails + {'a@example.com'} where user_id = 7b1e...;
update users set prefs['theme'] = 'dark' where user_id = 7b1e...;

-- Collections must be frozen to be a primary key or to be nested
create table user_tags (
  user_id uuid,
  tag_set frozen<set<text>>,
  primary key (user_id, tag_set)
);
```

Rules that save pain: collections are read and written whole, so a huge set is expensive on every access; a collection is stored as a single cell and is therefore capped at roughly 2 GB (a map is also limited to 65,535 keys); and an unfrozen collection cannot be indexed or used in a primary key. Use a separate table with a clustering column when the collection can grow unboundedly.

A non-frozen collection is stored as one cell, so every read and write touches the whole collection, and overwriting one element writes a tombstone for the old cell plus the new value. Elements have no individual timestamps. A `map` is limited to 65,535 keys and a collection cell to roughly 2 GB. Anything that can grow without bound belongs in its own table with a clustering column.

## Explain what is a Cluster in Cassandra?

A cluster is a container for keyspaces. Cassandra database is segmented over several machines that operate together. The cluster is the outermost container which arranges the nodes in a ring format and assigns data to them. These nodes have a replica which takes charge in case of data handling failure.

```bash
nodetool status            # UN/UJ/DN/LE states per node, with load and tokens
nodetool describecluster   # cluster name, snitch, schema version, partitioner
nodetool ring              # token ownership across the ring
```

There is no master node: every node gossips cluster state, and a coordinator is chosen per request (usually the client's contact node). Data distribution is by token range, and adding a node moves a fraction of the ranges to it, which is why throughput scales close to linearly until the client or the network is the bottleneck.

Membership runs on gossip. Nodes exchange state roughly every second, seed nodes bootstrap discovery, and the snitch reports each node's DC and rack. `nodetool status` shows `UN` (up and normal), `DN` (down), `UJ` (joining), and `UL` (leaving). A node stuck in `UJ` is streaming data and must not be restarted, because restarting throws away the bootstrap progress and can leave ranges half-owned.

## State the differences between Node and Cluster And DataCenter in Cassandra?

While a node is a single machine running Cassandra, cluster is a collection of nodes that have similar type of data grouped together. Data centers are useful components when serving customers in different geographical areas. You can group different nodes of a cluster into different data centers.

```text
Cluster: acme-prod
  DC: dc1 (Dallas)      -> racks rack1..rack3, RF 3
  DC: dc2 (Frankfurt)   -> racks rack1..rack3, RF 3
```

```bash
nodetool status            # shows DC and rack per node
nodetool gossipinfo        # what nodes know about each other
```

Data centers are the unit of replication topology: `NetworkTopologyStrategy` places RF replicas per DC, and `LOCAL_QUORUM` keeps a write inside one DC so transatlantic latency never enters the request path. Racks inside a DC are the unit of failure-domain awareness: replicas are spread across racks so one rack failure does not take out a quorum.

*Also asked as: What is node in Cassandra? What do you mean by data center in Cassandra? What are the differences between node and cluster and datacenter in Cassandra?*

Racks are failure domains, not physical requirements. With `NetworkTopologyStrategy` Cassandra walks the ring and prefers distinct racks, so with RF=3 across three racks one rack outage costs at most one replica per partition. That is exactly why `LOCAL_QUORUM` survives a rack failure: two replicas remain in the local DC and a quorum is two.

## What is data replication in Cassandra?

Data replication is the electronic copying of data from a database in one computer or server to a database in another so that all users can share the same level of information. Cassandra stores replicas on multiple nodes to ensure reliability and fault tolerance. The replication strategy decides the nodes where replicas are placed.

Replication Factor (RF) is the measure of the number of data copies that exist. RF=3 means three nodes hold each partition; those nodes are chosen by the snitch so that, with `NetworkTopologyStrategy`, copies land in different racks or data centers.

```sql
-- Three copies in dc1, three in dc2
alter keyspace iot
  with replication = {'class': 'NetworkTopologyStrategy', 'dc1': 3, 'dc2': 3};
```

```bash
nodetool getendpoints iot readings 's-1'   # which nodes hold this partition
nodetool repair -pr iot                    # repair the ranges this node owns
```

Practical rules: RF=3 tolerates one replica down for `QUORUM` operations and two for `ONE`; RF should be at least 3 in production; raising RF on a live cluster requires `nodetool repair` (or `rebuild` on a new DC) before the extra copies are usable.

*Also asked as: What is Replication Factor in Cassandra?*

`NetworkTopologyStrategy` places RF replicas per data center, preferring distinct racks. `SimpleStrategy` ignores DCs and racks and belongs only in a single-DC development cluster. `nodetool getendpoints <keyspace> <table> <partition-key>` prints the exact replicas for one key, which is the quickest way to demonstrate placement in an interview.

## Can we change Replication Factor on a Live Cluster?

Yes, but it will require running repair to alter the replica count of existing data. Changing the replication strategy in the keyspace metadata only changes where future writes are sent; existing data is streamed to the new replicas by `nodetool repair` (or `nodetool rebuild` when adding an entire data center).

```bash
# 1. Change the topology metadata
cqlsh -e "alter keyspace iot with replication = {'class':'NetworkTopologyStrategy','dc1':3,'dc2':3};"

# 2. Stream the missing replicas to the new nodes
nodetool repair -full iot

# 3. In a new DC, build from an existing DC instead of repairing
nodetool rebuild -- dc1
```

Do this during low traffic: repair and rebuild stream a lot of data and compete for disk and network. Over-replication (RF larger than the number of nodes in a DC) leads to duplicate copies on the same node and wasted space.

Changing RF changes where new writes go; existing data does not move. The extra replica is empty until it receives the data. `nodetool repair -full` streams the ranges the node now owns, or `nodetool rebuild -- <dc>` builds a brand-new data center from an existing one. Until that finishes, a read that happens to touch only the new replica plus one old one can still see stale data.

## Explain Cap Theorem?

With a strong requirement to scale systems when additional resources are needed, CAP Theorem plays a major role in maintaining the scaling strategy. It is an efficient way to handle scaling in distributed systems. Consistency, Availability and Partition tolerance (CAP) theorem states that in distributed systems like Cassandra, users can enjoy only two out of these three characteristics.

One of them needs to be sacrificed. Consistency guarantees the return of the most recent write for the client, Availability returns a rational response within minimum time, and in Partition Tolerance the system will continue its operations when network partitions occur. The two options available are AP and CP.

Cassandra is classically AP: the ring keeps serving reads and writes when nodes cannot reach each other, and the client chooses consistency per operation. A `QUORUM` write with `QUORUM` read gives strong consistency while a majority is reachable; when a partition removes a majority, those operations fail (unavailable) rather than returning stale-but-successful results. That single sentence - AP by default, CP per query when you ask for a quorum - is the answer interviewers are listening for.

PACELC is the sharper framing. If there is a Partition, choose Availability or Consistency; Else, in normal operation, choose Latency or Consistency. Cassandra is AP and, on the normal path, EL: you trade latency for consistency by raising the consistency level. A partition can also make quorum operations fail rather than return stale-but-successful results.

## What are consistency levels in Cassandra?

Cassandra lets clients choose consistency per read and write. `ONE` is faster and more available but can return stale data. `QUORUM` balances consistency and availability by requiring a majority of replicas. `ALL` is strongest but least available.

The levels you must know:

+ `ONE` / `TWO` / `THREE`: that many replicas must respond, fastest and least consistent.
+ `QUORUM`: a majority of all replicas (`floor(RF/2) + 1`), for example 2 of RF=3.
+ `LOCAL_QUORUM`: a majority of replicas in the coordinator's data center only.
+ `EACH_QUORUM`: a quorum in every data center (writes only).
+ `LOCAL_ONE`: one replica in the local data center.
+ `ALL`: every replica must respond; any node down fails the operation.
+ `ANY`: at least one node (including a hinted handoff) accepts the write; weakest write level.
+ `SERIAL` / `LOCAL_SERIAL`: Paxos consistency for lightweight transactions.

Example with replication factor 3:

```text
write consistency QUORUM = 2 replicas acknowledge
read consistency QUORUM = 2 replicas respond
```

When read quorum plus write quorum is greater than the replication factor (`R + W > N`), the read set and write set must overlap, so a read is guaranteed to see the latest acknowledged write. That is strong consistency, and the trade-off is higher latency and lower tolerance for replica outages.

```sql
-- Per-query consistency override
consistency local_quorum;
select * from orders_by_customer where customer_id = 'C1';
```

In [ ]:
# ILLUSTRATIVE - requires cassandra: install to run, or read on.
from cassandra import ConsistencyLevel
from cassandra.cluster import Cluster
from cassandra.query import SimpleStatement

cluster = Cluster(["10.0.0.11", "10.0.0.12", "10.0.0.13"])
session = cluster.connect("shop")

statement = SimpleStatement(
    "select * from orders_by_customer where customer_id = %s",
    consistency_level=ConsistencyLevel.LOCAL_QUORUM,
)
rows = session.execute(statement, ("C1",))

Reads that hit multiple replicas also run a read-repair (or a digest request when `read_repair_chance`/`dc_local_read_repair_chance` is enabled): the coordinator asks for the full data from one replica, asks another for a digest, and repairs the winner if they differ.

*Also asked as: What do you mean by consistency in Cassandra? Define the Consistency Levels for Read Operations in Cassandra?*

Quorum arithmetic: `QUORUM = floor(RF/2) + 1`, so RF=3 needs 2, RF=4 needs 3, and RF=5 needs 3. `LOCAL_QUORUM` counts only replicas in the coordinator's data center, so a two-DC RF=3 keyspace needs 2 in the local DC rather than a global majority. `EACH_QUORUM` requires a quorum in every DC and is valid for writes only.

Worked example: RF=3 with `W=QUORUM` and `R=QUORUM` gives 2 + 2 > 3, so every read set overlaps the last acknowledged write set and the read sees it. Common mistake: mixing levels across DCs. A write at `LOCAL_QUORUM` in dc1 and a read at global `QUORUM` may overlap only on dc2 replicas that never saw the write, so the guarantee does not hold across regions.

## Explain the concept of tunable consistency in Cassandra?

Tunable Consistency is a characteristic that makes Cassandra a favored database choice of Developers, Analysts and Big Data Architects. Consistency refers to the up-to-date and synchronized data rows on all their replicas. Cassandra's Tunable Consistency allows users to select the consistency level best suited to their use cases. It supports two consistencies - Eventual Consistency and Strong Consistency.

The former guarantees consistency when no new updates are made on a given data item: all accesses return the last updated value eventually. Systems with eventual consistency are known to have achieved replica convergence.

For strong consistency, Cassandra supports the following condition:

R + W > N, where

N - Number of replicas
W - Number of nodes that need to agree for a successful write
R - Number of nodes that need to agree for a successful read

```text
RF=3, W=ONE,   R=ONE   -> 1+1=2 <= 3, eventual consistency, fastest
RF=3, W=QUORUM,R=QUORUM-> 2+2=4 >  3, strong consistency, balanced
RF=3, W=ALL,   R=ONE   -> 3+1=4 >  3, strong but a single node failure blocks writes
RF=5, W=LOCAL_QUORUM, R=LOCAL_QUORUM in one DC -> 3+3>5 within the DC
```

Common pairings: `ONE`/`ONE` for high-volume metrics where a slightly stale reading is acceptable, `LOCAL_QUORUM`/`LOCAL_QUORUM` for user-facing data in a multi-DC deployment, and `QUORUM`/`ALL` or LWT for anything with a uniqueness or money invariant. Consistency is chosen per request, not per cluster, so a single keyspace can serve both.

Strong consistency comes from overlap, not from a single setting. `R + W > RF` guarantees the read set intersects the write set, so at least one replica in the read holds the newest write. Raising either side narrows the set of failures the operation can tolerate. `R=1, W=ALL` is strong but one failed write blocks everything; `QUORUM/QUORUM` keeps one failure of headroom on each side.

The same keyspace can serve both: a metrics dashboard reads at `ONE`, and a payment lookup reads at `LOCAL_QUORUM` in the same cluster. Consistency is chosen per statement by the client, so document the level next to each query rather than assuming a cluster-wide default.

## Does Cassandra support ACID transactions?

Unlike relational databases, Cassandra does not support full ACID transactions across partitions. It offers something narrower, and the precise answer matters:

- Atomicity and isolation are guaranteed at the single-partition level: a mutation to one partition is applied atomically, and each partition is updated in isolation.
- Durability is provided by the commit log plus replication.
- Consistency is configurable via consistency levels and, for compare-and-set semantics, lightweight transactions (LWT).
- Across partitions, there is no rollback. A `BATCH` is atomic in the sense that it will eventually apply, but a logged batch is not isolated and can partially apply if the coordinator dies mid-flight.

```sql
-- Lightweight transaction: Paxos, strong but 4x slower; use sparingly
insert into users_by_email (email, user_id) values ('a@example.com', 7b1e...)
  if not exists;

-- Condition on an existing row
update claims set status = 'assigned', owner = 'agent-7'
  where claim_id = 'c-1'
  if status = 'new';

-- Logged batch: multiple tables, same partition key where possible, atomic eventually
begin batch
  insert into users_by_id (user_id, email) values (7b1e..., 'a@example.com');
  insert into users_by_email (email, user_id) values ('a@example.com', 7b1e...);
apply batch;
```

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
result = session.execute(
    "insert into users_by_email (email, user_id) values (%s, %s) if not exists",
    ("a@example.com", user_id),
)
if not result.was_applied:
    existing = result.one()

Best practice: keep the invariant inside one partition so a single atomic mutation suffices. Use LWT when you must have uniqueness, and remember that a failed LWT is not an error - check `was_applied`.

LWT uses Paxos and costs about four round trips, so it is roughly an order of magnitude slower than a normal write and belongs only on real uniqueness invariants. A `SERIAL` read is required to see an in-progress Paxos transaction. A LWT returns `[applied]` and, on failure, the current row, so the application decides whether to retry. Logged batches are atomic but not isolated; an unlogged batch that spans partitions can partially apply.

## Explain how Cassandra delete data?

SSTables are immutable and cannot remove a row from SSTables. When a row needs to be deleted, Cassandra writes a special value called a Tombstone. When the data is read, the Tombstone value is considered as deleted. The tombstone is written with a timestamp and lives for `gc_grace_seconds` (default 10 days), after which compaction may discard it - but only if every replica has seen it.

```sql
-- Row delete: tombstone for the whole partition
delete from orders_by_customer where customer_id = 'C1';

-- Column delete: tombstone for one cell
delete amount from orders_by_customer where customer_id = 'C1' and order_ts = '...';

-- Range delete
delete from readings where sensor_id = 's-1' and ts < '2026-01-01T00:00:00Z';

-- TTL delete: cell expires instead of being explicitly deleted
insert into sessions (user_id, token, expires_at) values ('u-1', 'abc', toTimestamp(now()))
  using ttl 3600;
```

```bash
nodetool tablestats iot.readings | grep -i tombstone
nodetool garbagecollect iot readings     # drop tombstones past gc_grace_seconds
nodetool repair iot                      # essential before tombstones can be dropped
```

*Also asked as: How does Cassandra delete data?*

Tombstone kinds: a cell tombstone deletes one column, a row tombstone deletes one clustering row, a range tombstone deletes a slice of clustering rows, and a partition tombstone deletes the whole partition. TTL expiry is a local deletion time on the cell, not a background delete, so it behaves like a tombstone until compaction can drop it after `gc_grace_seconds`.

## Explain Tombstone in Cassandra?

Tombstone is a marker indicating a column or row deletion. These marked columns are deleted during compaction. Tombstones are of great significance as Cassandra supports eventual consistency, where a delete must be recorded rather than simply omitted, otherwise a delayed replica could resurrect the data.

The lifecycle and the failure mode to describe:

+ A tombstone has a timestamp; a later write with a higher timestamp can overwrite ("resurrect") the value - this is intentional.
+ Tombstones are kept for `gc_grace_seconds` (default 864000 = 10 days) to give every replica time to learn about the delete.
+ If a node has been down longer than `gc_grace_seconds` and repair did not run, compaction can drop the tombstone on the live replicas; when the stale node returns it still has the old value and can resurrect deleted data. That is why skipping repair for more than 10 days is dangerous.
+ Reading a range with many tombstones is slow: the coordinator must scan and filter all of them, and warnings appear in the logs (`Read ... tombstone cells`).

```sql
-- Lower gc_grace_seconds only for tables that are repaired frequently
alter table sessions with gc_grace_seconds = 86400;

-- Set a TTL so data expires instead of accumulating tombstones forever
insert into sessions (user_id, token) values ('u-1', 'abc') using ttl 3600;
```

```bash
# Tombstone counts per table and the fix
nodetool tablestats iot.sessions | grep -i tombstone
nodetool repair -full iot
nodetool garbagecollect iot sessions
```

*Also asked as: What is Tombstone in Cassandra?*

Read amplification from tombstones: a scan that walks 1,000,000 rows to return 10 live ones reads all the tombstones, and the coordinator holds them until it can prove they are deleted. Cassandra logs a warning above `tombstone_warn_threshold` (1000) and fails the query above `tombstone_failure_threshold` (100000). The fixes are fewer deletes (TTL plus TWCS), partition-scoped deletes, and running repair so `gc_grace_seconds` can safely pass.

## What is difference between Column and Super Column?

Both elements work on the principle of a tuple having name and value. However, the former's value is a string while the value in the latter is a Map of Columns with different data types. Unlike Columns, Super Columns do not contain the third component of timestamp.

```text
Column:       name -> (value, timestamp)
SuperColumn:  name -> { col1 -> (v1, ts1), col2 -> (v2, ts2), ... }
```

Super columns were a Thrift-era construct that nested a map of columns under one name; they were removed because the whole super column had to be read and written as one unit, and their inner columns shared a single row-level ordering. They are gone from modern Cassandra. The CQL equivalents are:

```sql
-- Replaces "super column" nesting: clustering columns give the inner map
create table user_contacts (
  user_id text,
  contact_type text,
  contact_value text,
  primary key ((user_id), contact_type)
);

-- Or collections / UDTs for small, bounded nested values
create type phone (country text, number text);
create table users (
  user_id uuid primary key,
  phones map<text, frozen<phone>>
);
```

*Also asked as: What is SuperColumn in Cassandra?*

This pair is historical. A super column was a nested map of columns under one name, and the whole super column had to be read and written as one unit. Modern CQL replaces it with clustering columns, which give an inner sorted set of rows, or with a collection for a small bounded nesting. Thrift, which defined the construct, was removed in Cassandra 4.0.

## Explain what is Cassandra cqlsh?

Cassandra cqlsh is a query language shell that enables users to communicate with its database. By using Cassandra cqlsh, you can do the following things:

+ Define a schema
+ Insert data and
+ Execute a query

Beyond DDL and DML, cqlsh is the operator's console: `DESCRIBE` prints schema, `COPY` moves CSV, `SOURCE` runs a file, `CAPTURE` records output, `CONSISTENCY` sets the level, and `TRACING` shows where a query spent its time.

```bash
cqlsh                                        # local node, default port 9042
cqlsh 10.0.0.11 9042 -u app -p secret -k iot
cqlsh --version                              # cqlsh version (also: cqlsh-version)
cqlsh -e "describe keyspace iot"
```

```text
cqlsh:iot> DESCRIBE TABLES;
cqlsh:iot> CONSISTENCY LOCAL_QUORUM;
cqlsh:iot> TRACING ON;
cqlsh:iot> SELECT * FROM readings WHERE sensor_id = 's-1' LIMIT 5;
cqlsh:iot> CAPTURE '/tmp/readings.out';
cqlsh:iot> SOURCE 'schema.cql';
cqlsh:iot> COPY readings TO 'readings.csv' WITH HEADER = TRUE;
```

*Also asked as: Which command is used to start the cqlsh prompt? What is the usage of cqlsh version command?*

`TRACING ON` is the fastest way to see the read path for one query: it names the coordinator, the replicas contacted, whether the Bloom filter and index were used, and where the time went. `DESCRIBE` and `system_schema.tables` show the schema, while `system_schema.columns` shows the key role (`partition_key`, `clustering`) of every column in a table.

## Mention what needs to be taken care while adding a Column?

+ While adding a column you need to take care that the column name is not conflicting with the existing column names.
+ The table is not defined with the compact storage option (obsolete in modern Cassandra).
+ The new column is always nullable: CQL has no `DEFAULT` or `NOT NULL` column constraint, so existing rows simply have no value for it.

```sql
alter table users add last_login timestamp;
alter table users add country text static;         -- static: one value per partition
alter table users add rating int;                  -- nullable; null until a write sets it

-- Remove a column (writes tombstones for existing data)
alter table users drop rating;
```

Adding a column is metadata-only and instant because Cassandra is schema-on-read: old SSTables simply do not contain it. Dropping a column creates tombstones and should be done during low traffic. Changing a column's type is only allowed for compatible widenings.

Adding a column is metadata-only and instant because old SSTables simply do not contain it; the value reads back as null. Dropping a column writes tombstones for existing data and belongs in a low-traffic window. Changing a type is limited to compatible widenings such as `int` to `bigint`.

## What are the trade-offs and when should you not use Cassandra?

- Consistency vs availability: `LOCAL_QUORUM` reads and writes give strong consistency inside a DC but require a majority of replicas to be up, so a rack failure can block writes. `ONE` keeps serving during failures but returns stale data and, with hinted handoff, can acknowledge a write that no live replica holds yet. Pick per query, and document the choice.
- Eventual consistency and reconciliation: replicas converge only after read repair, hinted handoff, or `nodetool repair`. If repair does not run within `gc_grace_seconds`, deleted data can be resurrected. Repair is not optional maintenance - it is part of the consistency model.
- Hot partitions: partitioning by a low-cardinality key (`status`, `country`) or by a monotonically increasing key (`ts` alone) concentrates load on one node and one token range. There is no query planner to save you; a partition is served by a single node, so a single partition can never be read faster than one machine allows.
- Unbounded partitions: a partition that grows forever (all events of one user) eventually exceeds the practical size limit, triggers GC pressure, and makes reads slow. Bucket by time (`primary key ((user_id, month), ts)`) to keep partitions bounded.
- Write amplification: every write goes to the commit log and the memtable, then is rewritten during compaction. Tuning for read latency (leveled compaction) increases write amplification, and vice versa. A high read_repair or heavy tombstone load multiplies the effects.
- No joins, no ad-hoc queries, no aggregations at scale: every new query shape usually needs a new table and a backfill. `ALLOW FILTERING` scans the cluster. Standard `count(*)` on a large table is a full scan.
- When NOT to use Cassandra: strong multi-row transactional systems (ledgers that must commit across entities), workloads dominated by ad-hoc analytical queries and joins, small datasets (a single PostgreSQL instance is simpler and faster), read-heavy workloads with flexible filter patterns, and anything where the team cannot commit to running repair, monitoring compaction, and designing a table per query.

A compact decision checklist: the partition key is chosen from query patterns and cardinality; every query is single-partition or a bounded fan-out; the consistency level is documented per query; repair runs well inside `gc_grace_seconds`; the compaction strategy matches the write, read, and TTL pattern; and there is a plan for hot partitions. If a workload cannot be expressed as partition-scoped reads and writes, Cassandra is the wrong tool.

## How do you partition for even distribution and find a hot partition?

Even distribution starts with the partition key. The token space is fixed, so balance comes from having many distinct keys that hash uniformly, not from having many nodes. A partition is served by one replica set, so one very large or very busy partition puts all of its load on the same nodes no matter how large the cluster is.

Diagnosis is a measurement loop, not intuition:

```bash
nodetool status                          # load per node: an uneven Load column is the first clue
nodetool ring                            # token ownership across the cluster
nodetool tablestats iot.readings         # partition count, SSTable count, tombstones, read/write latency
nodetool tablehistograms iot.readings    # p50/p95/p99 read and write latency for one table
nodetool tpstats                         # dropped mutations and pending tasks per stage
nodetool getendpoints iot readings 's-1' # which nodes serve one specific key
```

Read and write latency live per table: `Local read latency` and `Local write latency` in `nodetool tablestats`, and the `ReadLatency` and `WriteLatency` JMX metrics for p99. A hot partition shows up as one node with much higher load, a p99 rising while p50 stays flat, and drops in the mutation stages of `tpstats`.

Worked example: a table keyed by `country` has 4 distinct values and 500 million rows, so roughly 125 million rows land on one replica set and every read for that country hits the same nodes. Adding a high-cardinality component (`(country, user_id)`) or a time bucket fixes placement. Common mistake: salting every key to force balance and then being unable to read a partition without scanning all salts; salt only monotonic keys, and keep the salt count small enough to fan out cheaply.

## How do you reason about consistency versus latency?

The quorum condition `R + W > RF` is the whole consistency model. With RF=3: `ONE/ONE` gives 2 <= 3 and eventual consistency; `QUORUM/QUORUM` gives 4 > 3 and strong consistency; `TWO/TWO` gives 4 > 3 and tolerates one failure on each side; `ONE/ALL` gives 4 > 3 but a single down replica blocks every write.

Latency grows with the number of replicas that must respond, because the coordinator waits for the slowest of them. `ONE` waits for the fastest replica, `QUORUM` for the second fastest of three, and `ALL` for the slowest. Raising a level buys consistency and fault tolerance and pays in tail latency, so the right level is the weakest one that still satisfies the invariant.

Worked example: RF=3 in one DC with read p99 of 4 ms and write p99 of 5 ms at `LOCAL_QUORUM`. Moving reads to `LOCAL_ONE` lowers read latency but allows stale rows after a failed write, which is fine for a metrics dashboard and wrong for an account balance. Common mistake: using global `QUORUM` in a two-DC cluster for a user-facing read, which counts replicas in the remote DC and puts a cross-region round trip on the latency path. Use `LOCAL_QUORUM`.

## How do you choose a compaction strategy and manage tombstones?

Choose the strategy from the workload. Write-heavy with no time locality favors STCS. Read-heavy point lookups with enough IOPS favor LCS. Time series with TTL favors TWCS. New clusters on Cassandra 5.0 can start with UCS and let it adapt. The signals that the choice is wrong are in `nodetool tablestats` (SSTable count, partition size, tombstones) and `nodetool compactionstats` (pending compactions and throughput).

```sql
-- Time series: TTL expiry drops whole windows
alter table iot.readings_by_sensor_day
  with compaction = {'class': 'TimeWindowCompactionStrategy',
                     'compaction_window_unit': 'DAYS',
                     'compaction_window_size': 1}
  and gc_grace_seconds = 864000;

-- Read-heavy point lookups: fewer SSTables per read, more write amplification
alter table chat.messages_by_conversation
  with compaction = {'class': 'LeveledCompactionStrategy',
                     'sstable_size_in_mb': 160};
```

`gc_grace_seconds` is a correctness deadline, not a cleanup delay. A tombstone can be purged only after every replica has seen it, and repair is what guarantees that. If a node is down longer than `gc_grace_seconds` with no repair, its stale copy can resurrect deleted data. Lower the value only when repair is frequent and reliable. When tombstone reads are slow, prefer TTL expiry plus TWCS over explicit deletes.

```bash
nodetool tablestats iot.sessions | grep -i tombstone
nodetool compactionstats
nodetool garbagecollect iot sessions
nodetool repair -pr iot
```

## How do you tune the JVM, memtable flush, and SSTable sizing?

Keep the heap moderate and let off-heap structures do the work. A large heap lengthens GC pauses, and every request on that node waits during a pause. Bloom filters, the row cache, and the chunk cache live off-heap, and the memtable is off-heap by default. With the default G1 collector, watch pause time and pause frequency, not the fraction of heap used.

Flush pressure comes from write rate and memtable budget. `memtable_heap_space_in_mb` and `memtable_offheap_space_in_mb` set the budget, and Cassandra flushes the largest memtables when `memtable_cleanup_threshold` is exceeded. Flushing too often creates many small SSTables and raises read amplification; flushing too rarely makes commit log replay long after a crash. Aim for a small, stable number of SSTables per table.

```bash
nodetool info                 # heap used, off-heap, GC counts and pause times
nodetool gcstats              # GC pause history
nodetool tablestats iot.readings | grep -iE 'memtable|SSTable|compacted partition'
nodetool flush iot readings   # force memtable -> SSTable while tuning
nodetool setcompactionthroughput 256
```

SSTable sizing: single digits of SSTables per table is healthy for read-heavy tables, dozens mean compaction is behind, and hundreds mean every read pays for the merge across candidates. For LCS, `sstable_size_in_mb` controls the target size of upper levels, and 160 MB is the common default. Common mistake: sizing the heap as a fraction of total RAM and ignoring off-heap needs, so Bloom filters and memtables push the node into swap and latency collapses.

## Things to remember: key numbers and limits


| Topic | Fact | Why it matters |
|---|---|---|
| Partitioning | the partition key is hashed (Murmur3) to a token; each node owns token ranges | you query by partition key or you scan — design tables per query |
| Replication factor | RF = copies per key, datacenter-aware with NetworkTopologyStrategy in production | RF=3 per DC is the common baseline |
| Consistency level | R + W > RF gives strong consistency (e.g. QUORUM/QUORUM at RF=3) | ONE is fastest but eventually consistent; ALL is fragile |
| Lightweight transactions | LWT (IF EXISTS / IF NOT EXISTS) uses Paxos — several round trips | fine for rare checks, wrong for per-write paths |
| Tombstones | a delete writes a marker, removed after `gc_grace_seconds` (10 days default) | heavy delete workloads accumulate tombstones that slow reads |
| Compaction | STCS (default), LCS (read-heavy, stable sizes), TWCS (time series) | wrong strategy = read amplification and disk churn |
| Partition size | large partitions (unbounded clustering rows) degrade; keep partitions bounded | buckets (truncated dates) contain growth |
| ALLOW FILTERING | disables the index discipline — a full scan | a query that needs it means the table is modeled for the wrong question |


## Things to remember: semantics to memorize


* **Model the query first, then the table.** Cassandra tables are materializations of access patterns; one logical entity often becomes several physical tables.
* **A primary key = partition key + clustering columns.** The partition key picks the nodes; clustering columns define the on-disk order inside the partition, which is what makes range reads possible.
* **Write path: commit log + memtable, flush to SSTables.** Writes are cheap and durable; reads merge memtable + SSTables, so read cost grows with SSTable count until compaction.
* **Tombstones are data.** Range deletes and TTLs write markers; a read scanning thousands of tombstones fails with tombstone-overflow warnings long before disk runs out.
* **CAP answer: AP with tunable C.** Consistency level per query is the tuning knob; LWT is the escape hatch for linearizable single-partition writes.


## Things to remember: mistakes that keep coming back


* Designing the table from the entity model instead of the query list — the 'one table per entity' instinct is the fastest way to full scans.
* Low-cardinality partition keys (status, boolean flags) — a few hot partitions carry the cluster.
* Unbounded time-series rows in one partition — bucket by day/hour so a partition never grows past a comfortable size.
* Deleting heavily without adjusting compaction or gc_grace — tombstones pile up and reads slow weeks before anyone connects the cause.
* LWT on a hot write path — Paxos round trips under load; use idempotent upserts where possible.
* ALLOW FILTERING in production code — remodel the table instead; the query wants its own table or a materialized view.
* Running SimpleStrategy across datacenters — NetworkTopologyStrategy is the production strategy; SimpleStrategy puts replicas unpredictably and is for local testing.


## Real world use cases for Cassandra

### 1. IoT sensor telemetry

Business context: an industrial monitoring platform with 40,000 sensors writing every 10 seconds (~345M rows/day, ~126B rows/year before TTL). Access patterns: last 24 hours for one sensor for dashboards, last 30 days for one sensor for anomaly review, and almost never cross-sensor queries. Retention is 90 days with TTL.

Schema design: partition by `(sensor_id, day)` so each partition is bounded and one day of one sensor (8,640 rows) is a cheap range scan; cluster by `ts` descending. TWCS with a 1-day window means expired windows are dropped whole.

```sql
create keyspace iot
  with replication = {'class': 'NetworkTopologyStrategy', 'dc1': 3};

create table iot.readings_by_sensor_day (
  sensor_id text,
  day date,
  ts timestamp,
  value double,
  quality tinyint,
  primary key ((sensor_id, day), ts)
) with clustering order by (ts desc)
  and default_time_to_live = 7776000
  and compaction = {
    'class': 'TimeWindowCompactionStrategy',
    'compaction_window_unit': 'DAYS',
    'compaction_window_size': 1
  };

-- Anomaly review: last 30 days, one sensor, newest first
select ts, value, quality
from iot.readings_by_sensor_day
where sensor_id = 's-1042'
  and day in ('2026-10-04', '2026-10-03', '2026-10-02')
order by ts desc
limit 1000;
```

In [ ]:
# ILLUSTRATIVE - requires cassandra: install to run, or read on.
from datetime import datetime, timezone
from cassandra.cluster import Cluster

cluster = Cluster(["10.0.0.11", "10.0.0.12", "10.0.0.13"])
session = cluster.connect("iot")

insert_reading = session.prepare(
    "insert into readings_by_sensor_day (sensor_id, day, ts, value, quality) "
    "values (?, ?, ?, ?, ?)"
)
now = datetime.now(timezone.utc)
session.execute(insert_reading, ("s-1042", now.date(), now, 21.7, 0))

Why this design: writes are pure appends spread evenly across the ring; every read touches exactly one partition per day and is served by one node; TTL plus TWCS removes old data without a delete job. The partitioning key is high-cardinality and non-monotonic, so no hot node.

### 2. User activity feed and inbox

Business context: a messaging product with 200M users; the inbox for a user must load in under 50 ms, and messages arrive at ~50k/s at peak (about 4B/day). Access patterns: newest N messages in one conversation, mark-as-read, and unread counts. There is no need to query "all messages by sender" online.

Schema design: one table per query. `messages_by_conversation` partitions by `(conversation_id, bucket)` with `bucket` derived from the month or a bucket id to bound partition size; cluster by `message_id` (a `timeuuid`) descending so the newest messages sort naturally.

```sql
create table chat.messages_by_conversation (
  conversation_id uuid,
  bucket int,                       -- e.g. year*100 + month, bounds the partition
  message_id timeuuid,
  sender_id uuid,
  body text,
  primary key ((conversation_id, bucket), message_id)
) with clustering order by (message_id desc)
  and compaction = {'class': 'TimeWindowCompactionStrategy',
                    'compaction_window_unit': 'MONTHS',
                    'compaction_window_size': 1};

-- Denormalized second table for the same event, queried by user
create table chat.conversations_by_user (
  user_id uuid,
  last_message_at timestamp,
  conversation_id uuid,
  unread int,
  primary key ((user_id), last_message_at, conversation_id)
) with clustering order by (last_message_at desc);
```

In [ ]:
# ILLUSTRATIVE - requires cassandra: install to run, or read on.
from cassandra.util import uuid_from_time
from datetime import datetime, timezone

message_id = uuid_from_time(datetime.now(timezone.utc))
session.execute(
    "insert into chat.messages_by_conversation "
    "(conversation_id, bucket, message_id, sender_id, body) values (%s, %s, %s, %s, %s)",
    (conversation_id, 202610, message_id, sender_id, body),
)
# Unread counter for the recipient, same logical event, second table
session.execute(
    "update chat.conversations_by_user set unread = unread + 1, "
    "last_message_at = %s where user_id = %s and last_message_at = %s and conversation_id = %s",
    (datetime.now(timezone.utc), recipient_id, last_message_at, conversation_id),
)

Why this design: reads are single-partition and never join; writes are duplicated into the tables that serve the two known queries. Bucketing keeps a very chatty conversation from creating an unbounded partition. The trade-off is that the application owns both writes - there is no join to reconstruct the second view.

### 3. Multi-tenant SaaS event store

Business context: a B2B analytics SaaS ingesting 5,000 events/s across 2,000 tenants, with one tenant producing 40% of the volume. Access patterns: recent events for one tenant (filtered by event type), hourly rollups per tenant, and export to a warehouse nightly. Retention is 30 days in Cassandra and everything in S3.

Schema design: partition by `(tenant_id, hour)` so even the largest tenant is spread across many partitions and every read is tenant-scoped; cluster by `event_ts` and `event_id`, and keep a `day` rollup table for dashboards. The big tenant's volume is handled by partition granularity rather than by a special case.

```sql
create keyspace saas
  with replication = {'class': 'NetworkTopologyStrategy', 'dc1': 3, 'dc2': 3};

create table saas.events_by_tenant_hour (
  tenant_id text,
  hour timestamp,
  event_ts timestamp,
  event_id timeuuid,
  event_type text,
  payload text,
  primary key ((tenant_id, hour), event_ts, event_id)
) with clustering order by (event_ts desc)
  and default_time_to_live = 2592000;

create table saas.event_counts_by_tenant_day (
  tenant_id text,
  day date,
  event_type text,
  count counter,
  primary key ((tenant_id, day), event_type)
);
```

```sql
-- Update a counter table (counters are their own table type)
update saas.event_counts_by_tenant_day
  set count = count + 1
  where tenant_id = 't-88' and day = '2026-10-04' and event_type = 'checkout';

-- Dashboard query for one tenant, one hour
select event_type, event_ts, payload
from saas.events_by_tenant_hour
where tenant_id = 't-88' and hour = '2026-10-04T14:00:00Z'
  and event_ts >= '2026-10-04T14:00:00Z'
limit 200;
```

Why this design: hour-bucketed partitions stay bounded no matter how large a tenant grows, and `LOCAL_QUORUM` in each data center keeps tenant latency local. Counters are a native Cassandra feature with a caveat: counter updates are not idempotent under retries, so the safer pattern for exact counts is to read from the events table or compute rollups in Spark and write the result with normal `update ... set count = ?`.

### 4. Fintech ledger and idempotent writes

Business context: a payments ledger appending double-entry records at 5,000 writes/s with a strict rule that a retried request must never post twice. Access patterns: all entries for one account ordered by time, lookup by transaction id, and daily reconciliation by account and day.

Schema design: partition by `(account_id, month)` to bound history per account, cluster by `entry_ts` and `entry_id`; a separate table unique on `transaction_id` implements the idempotency guarantee with a lightweight transaction; every write uses `LOCAL_QUORUM` with `SERIAL` for the LWT.

```sql
create table ledger.entries_by_account_month (
  account_id text,
  month text,                         -- '2026-10'
  entry_ts timestamp,
  entry_id timeuuid,
  amount decimal,
  direction text,
  transaction_id text,
  primary key ((account_id, month), entry_ts, entry_id)
) with clustering order by (entry_ts desc);

create table ledger.txn_by_id (
  transaction_id text primary key,
  account_id text,
  amount decimal,
  posted_at timestamp,
  status text
);

-- Idempotency guard: only the first attempt succeeds
insert into ledger.txn_by_id (transaction_id, account_id, amount, posted_at, status)
  values ('txn-7f3a', 'acct-1', 5000, toTimestamp(now()), 'posted')
  if not exists;
```

Why this design: the write path is append-only and the LWT gives a real compare-and-set on the idempotency key. `LOCAL_QUORUM` (or `QUORUM` when a single DC) makes the acknowledged write durable on a majority, and `SERIAL` consistency around the LWT prevents two coordinators from both "winning". A ledger is exactly the case where you should be honest that Cassandra gives per-partition atomicity, not multi-table ACID: the debit and credit entries must be designed so the invariant can be checked and the LWT outcome inspected (`was_applied`).

### 5. Time-ordered log and metrics ingestion

Business context: an observability platform ingesting 2 TB/day of application logs and infrastructure metrics. Access patterns: search one service's logs for a short time window, read the latest metrics for a host, and run occasional bulk aggregation in Spark. Old data is moved to object storage after 14 days.

Schema design: partition by `(service, day)` for logs with TWCS, and by `(host_id, day)` for metrics; cluster by timestamp with a `timeuuid` tiebreaker so identical timestamps do not overwrite each other. Compression is tuned up (Zstd) because log lines are highly repetitive.

```sql
create table logs.entries_by_service_day (
  service text,
  day date,
  ts timestamp,
  log_id timeuuid,
  level text,
  message text,
  primary key ((service, day), ts, log_id)
) with clustering order by (ts desc)
  and default_time_to_live = 1209600
  and compression = {'class': 'ZstdCompressor', 'chunk_length_in_kb': 64}
  and compaction = {'class': 'TimeWindowCompactionStrategy',
                    'compaction_window_unit': 'DAYS',
                    'compaction_window_size': 1};

-- Search window for one service, newest first
select ts, level, message
from logs.entries_by_service_day
where service = 'checkout' and day = '2026-10-04'
  and ts >= '2026-10-04T13:00:00Z' and ts < '2026-10-04T14:00:00Z'
order by ts desc
limit 500;
```

```bash
# Operate a write-heavy cluster: watch compaction, repair, and latency
nodetool tpstats                 # dropped mutations, pending compactions by stage
nodetool compactionstats
nodetool proxyhistograms         # read/write latency percentiles
nodetool repair -pr iot          # incremental, primary-range repair on a schedule
nodetool tablestats logs.entries_by_service_day
```

Why this design: every query is scoped to a service and a day, so it is served from one partition per day and can be parallelized across days. `timeuuid` as the final clustering column guarantees that two log lines with the same millisecond are distinct rows rather than an overwrite. TWCS plus TTL means the oldest window is dropped as a unit, and Zstd keeps the on-disk footprint manageable for a workload where compression is a direct cost lever.

## Cassandra exercises

These exercises use only the Python standard library and model Cassandra placement and tuning arithmetic with small, runnable calculations. Each one names the task, gives a hint, states the expected result, then prints or asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The exercises reuse the same units the database reports: partitions and tokens for placement, bytes for partition sizing, SSTable counts for read amplification, and seconds for repair windows. The quorum calculator uses the real rule `QUORUM = floor(RF/2) + 1`; the partition sizer uses 1 MiB = 1,048,576 bytes; and the repair estimate assumes a 100 MB/s streaming rate, which is a realistic modern baseline.

### Try it 1. Choose a partition key from query patterns

**Task.** Given candidate partition keys with their cardinality and the query's equality and range filters, pick the key that satisfies the query and has the highest cardinality. Fall back to the next candidate if the best one fails the query.

**Hint.** A candidate is usable when every equality filter column appears in the partition key. Cardinality then decides the winner.

**Expected result.** `customer_id` is chosen: it covers the query and has 50,000 distinct values.

In [ ]:
candidates = {
    "status": ["active", "closed"],                          # cardinality 2
    "country": ["US", "DE", "IN", "BR"],                     # cardinality 4
    "customer_id": [f"c-{i:05d}" for i in range(50_000)],    # cardinality 50_000
}
query_equality = ["customer_id"]
query_range = ["order_ts"]

usable = {
    name: len(values)
    for name, values in candidates.items()
    if all(col in name for col in query_equality)
}
chosen = max(usable, key=usable.get)

print("usable keys:", usable)
print("chosen partition key:", chosen)
print("cardinality:", usable[chosen])

assert chosen == "customer_id"
assert usable[chosen] == 50_000

### Try it 2. Compute replica placement consistency (R + W > RF)

**Task.** For RF=3, enumerate every read/write consistency pair and classify each as strong or eventual. Then keep the strong pairs that also tolerate one failed replica on each side.

**Hint.** Strong means `R + W > RF`. A pair tolerates one failed replica on each side when both `W` and `R` are at most `RF - 1`.

**Expected result.** 9 of 9 combinations are listed, 6 are strong, and only `QUORUM/QUORUM` is strong while still tolerating one failure on each side.

In [ ]:
RF = 3


def quorum(n):
    return n // 2 + 1


pairs = [(w, r) for w in range(1, RF + 1) for r in range(1, RF + 1)]
strong = [(w, r) for w, r in pairs if w + r > RF]
tolerant = [(w, r) for w, r in strong if w <= RF - 1 and r <= RF - 1]

print(f"QUORUM for RF={RF} is {quorum(RF)}")
print("pairs:", pairs)
print("strong (R + W > RF):", strong)
print("strong and tolerates one failure each side:", tolerant)

assert len(pairs) == 9
assert len(strong) == 6
assert tolerant == [(2, 2)]
assert quorum(5) == 3

### Try it 3. Detect a hot partition from row counts

**Task.** Given the row count per partition, compute each partition's share of the table and flag any partition holding more than 40 percent of all rows.

**Hint.** Share is `rows / total_rows`. A single partition that owns most of the table is a hot partition even if the key looks reasonable.

**Expected result.** `US` is flagged with a share above 0.9.

In [ ]:
rows_per_partition = {
    "US": 4_100_000,
    "DE": 180_000,
    "IN": 210_000,
    "BR": 195_000,
    "MX": 205_000,
}
total = sum(rows_per_partition.values())
shares = {name: rows / total for name, rows in rows_per_partition.items()}
hot = {name for name, share in shares.items() if share > 0.40}

print(f"total rows = {total:,}")
for name, share in sorted(shares.items(), key=lambda kv: -kv[1]):
    print(f"  {name}: {share:.3f}")
print("hot partitions:", sorted(hot))

assert hot == {"US"}
assert round(max(shares.values()), 3) == 0.838

### Try it 4. Size a partition by row width and target 100 MB

**Task.** A sensor writes one row every 10 seconds, so 8,640 rows per sensor per day. Each row is 220 bytes. Compute how many rows fit in a 100 MB partition and how many days one `(sensor_id, day)` partition would take to reach that size.

**Hint.** Use 1 MiB = 1,048,576 bytes. Rows per partition is integer division; days to fill is `max_rows // rows_per_day`.

**Expected result.** About 476,625 rows fit in 100 MB, and one daily partition holds a small fraction of that, so `(sensor_id, day)` is bounded and safe.

In [ ]:
ROW_BYTES = 220
TARGET_MB = 100
MIB = 1024 * 1024
ROWS_PER_DAY = 8640

max_rows = TARGET_MB * MIB // ROW_BYTES
daily_rows = ROWS_PER_DAY
days_to_fill = max_rows // daily_rows
daily_mb = daily_rows * ROW_BYTES / MIB

print(f"rows per 100 MB partition: {max_rows:,}")
print(f"daily partition rows: {daily_rows:,} ({daily_mb:.2f} MiB)")
print(f"days to fill one partition: {days_to_fill}")

assert max_rows == 476_625
assert days_to_fill == 55
assert daily_mb < TARGET_MB

### Try it 5. Plan compaction from SSTable counts

**Task.** Given the current SSTable count per table, compute the read amplification of a point read as `1 memtable + N SSTables` and flag tables above 20 candidates. Then say which tables need compaction.

**Hint.** A read must consult the memtable and every SSTable that the Bloom filter did not rule out. Fewer SSTables means a shorter merge.

**Expected result.** `readings` and `events` are flagged, both far above the threshold, and `sessions` is healthy.

In [ ]:
sstables = {"readings": 412, "sessions": 6, "events": 48}
MEMTABLE = 1
MAX_CANDIDATES = 20

amplification = {t: n + MEMTABLE for t, n in sstables.items()}
need_compaction = {t: a for t, a in amplification.items() if a > MAX_CANDIDATES}

print("read amplification (candidates per point read):", amplification)
print("need compaction:", need_compaction)

assert need_compaction == {"readings": 413, "events": 49}
assert "sessions" not in need_compaction

### Try it 6. Compute tombstone read amplification

**Task.** A range scan reads 250,000 cells to return 120 live rows. Compute the tombstone count and the read amplification, and decide whether the scan crosses the default warn and failure thresholds.

**Hint.** `gc_grace_seconds` keeps tombstones alive, so each one must be read and discarded. The defaults are `tombstone_warn_threshold = 1000` and `tombstone_failure_threshold = 100000`.

**Expected result.** 249,880 tombstones, amplification about 2,083x, above both the warn and failure thresholds.

In [ ]:
rows_scanned = 250_000
live_rows = 120
WARN = 1000
FAILURE = 100_000

tombstones = rows_scanned - live_rows
amplification = rows_scanned / live_rows
status = "fail" if tombstones > FAILURE else ("warn" if tombstones > WARN else "ok")

print(f"tombstones = {tombstones:,}")
print(f"read amplification = {amplification:,.1f}x")
print("status:", status)

assert tombstones == 249_880
assert round(amplification, 1) == 2083.3
assert status == "fail"
assert tombstones > FAILURE > WARN

### Try it 7. Estimate repair scope

**Task.** A cluster has 12 nodes, RF=3, and 24 TiB of logical data. `nodetool repair -pr` on each node repairs only the ranges that node is primary for. Estimate the data streamed by a full ring repair and how long it takes at 100 MB/s of streaming throughput, then check it against the 10-day `gc_grace_seconds` window.

**Hint.** The dataset is stored RF times on disk, but a full ring repair streams each partition once. Time is `bytes / throughput`.

**Expected result.** 72 TiB on disk but about 24 TiB streamed, roughly 69.9 hours, comfortably inside the 240-hour `gc_grace_seconds` window.

In [ ]:
data_tib = 24
nodes = 12
rf = 3
throughput_mb_s = 100
GC_GRACE_HOURS = 10 * 24

on_disk_tib = data_tib * rf
per_node_primary_tib = data_tib / nodes
full_ring_tib = per_node_primary_tib * nodes
seconds = full_ring_tib * 1024 * 1024 / throughput_mb_s
hours = seconds / 3600

print(f"on-disk footprint at RF={rf}: {on_disk_tib} TiB")
print(f"primary data per node: {per_node_primary_tib:.2f} TiB")
print(f"full ring repair streams: {full_ring_tib:.2f} TiB")
print(f"estimated time at {throughput_mb_s} MB/s: {hours:.1f} hours")
print(f"gc_grace_seconds window: {GC_GRACE_HOURS} hours")

assert on_disk_tib == 72
assert full_ring_tib == data_tib
assert round(hours, 1) == 69.9
assert hours < GC_GRACE_HOURS